## Workflow Documentation

## Abstract

In this project I explored data from CARB data, most notably CalEnviroScreen 4.0 (CES) and CAPP Incentives (CAPPIs), to identify areas of over- and under-service based on state funding for CAPPIs. I looked closely at the South LA and San Jaquin Valley regions as they consistently score high on CES and have seen a lot of money aimed at improving air quality in the region.

Before even working with the data, I took a lot of time to familiarize myself with environmental goals in my regions of interest and different CAPP incentive types as they pertain to their project locations.

Next (after some data scrubbing), I cross-referenced CES score with CAPP funding sums on a per-tract basis to identify tracts with over- and under- funding. Of course, this begged the question of how far-reaching an incentive can be; as an exercise to estimate this, I pulled numbers from a CARB press release and concluded an average "dollar to impact a person" amount of $125/person.

With a dollar per person impacted number, I took the top 1% of CAPPIs by fund amount as a basis for geoprocessing (~\\$750,000); based on my //$125 assumption, this much funding should impact 6,000 people, which is a close average for the number of people residing in any single tract. Furthermore I concluded that CAPPIS in the bottom 99% have a reach of just the tract they were completed in.

For the top 1% of CAPPIs, I wrote an iterative loop to create an impact radius based on funding amount, assuming tracts are equally distributed in population. Radii tended to be large in the San Jaquin Valley (dispersed population), and small in South LA (dense population). This suggests that CAPP funding distribution does not properly account for the huge population density of LA, despite tracts in the area scoring extremely high on CES. To be clear, this is a terrible way to measure impact; most notably, all on-road emissions are not stationary sources, so any CAPPIs addressing these emissions realistically impact driving route corridors, not circles centered on the "location" of the project. However, this analysis is just for exercise, so I'm happy with the methods.

The results warranted further research and discussion, and I ended up deep in a 2010 paper describing dispersion modeling for pollution sources in the port of LA, concluding that diesel engines both onroad and on...ship, are the primary contributors to poor air quality in the area. This has been addressed extensively; ship emissions have been curbed through shore power initiatives, road emissions through a long list of vehicle electrification initiatives, including the Port of Long Beach's Clean Truck Program.

CES's most recent data shows that despite these investments, the South LA area continues to be a challenge for air quality engineering and environmental. This is likely due to the sheer number of emission sources, as well as the continued prevalence of longstanding socioeconomic issues in the area.

### Setup and Initial Thoughts

First thing's first, I understand air permits in California are issued by individal Air Districts, and there are 35 of them. I loaded in a layer from CARB and it looks like there are 47 polygons, so I'm going to dissolve them by district name to get my 35 districts.

I kept all the extra fields from the layer, using the "First" statistic after confirming that all IDs sharing an air district name also share all other attributes (except shape, obviously).

Next I spent a lot of time learning about CalEnviroScreen; I happened upon this dataset and who knew, it's really important. I've worked a bit in environmental justice in Colorado and am super interested to see what's going on here in California. I ended up symbolizing the results in 10 classes based on percentile, so I can easily see where I would expect CARB to be investing.

Next I discovered the California Air Resources Board GIS Portal, and one of the first datasets I saw was for Community Air Protection Program Incentives. I pulled the 2025 data, and good news, it looks like the distribution of incentives do roughly follow the CalEnviroScreen results distribution.

There's a lot to look into here. The obvious first question is this: How well does investment match EnviroScreen results? Ideas, ideas, ideas. Here's one, what if I summed investment in dollars for each CAPP tract, then calculated percentiles and compared to EnviroScreen precentiles. I could even calculate a ratio and create a new layer showing discrepancies.

### Geodatabase Management

This sounds like something the "Summarize Within" tool can help me with. Upon opening the tool I'm realizing that my geodatabase needs some work - I copied my water sandbox project to this air sandbox project file, and it looks like pro is bad at using "save as". This is a good opportunity to practice geodatabase management.

Problem 1: references to my water sandbox project, and the copy of my water sandbox project. Python says my default gdb env workspace is still my water sandbox copy, so I went ahead and changed those to the current project's gdb. I removed the water sandbox project gdb from the project, moved my "dissolve" result layer from the water sandbox copy gdb to my air quality sandbox gdb, and removed the water sandbox copy gdb from the project.

Problem 2: My CalEnviroScreen results and CCAPP initiatives don't show in my catalog. Looks like....the CCAPP initiatives are a live connection to CARB's GIS Portal. This may normally be a good thing, as updated data would be pulled automatically from the layer. However, since this project is mostly just a sandbox for me to play with analysis tools, I'll export it to my project. This is a good time to set up a second gdb, the first being for my created layers and the new one being for source data.

### Initial Project Scope, and the San Jaquin Valley

After fixing all the geodatabase issues, I can focus on data analysis. Here are some steps I took:

1. I used the summarize within to pull CAPP investment numbers into a new enviroscreen-CAPP summary layer.

2. I created a new field in my summary layer for a percentile calculation, and wrote some python to calculate this. I then subtracted the CIscore percentile from this number to get a percentile difference representing over- or under-investment in the community.

3. I created new layers for tracts with zero and non-zero investment, just to see what kind of land coverage these investments are getting. An obvious conclusion here is just how much of this incentive program has been used in the San Jaquin Valley (shoutout Fresno). Looking at the incentive types in the area confirms that this is a huge agricultural hub in California (I'm assuming THE agricultural hub).

The incentives in the San Jaquin Valley make a lot of sense; in the more rural areas, they focus on pollution reduction (agricultural burning, road emissions, etc.) in air that presumably settles into the center of the valley, where incentives shift quickly toward residential air filtering. This explains the sheer number of points around the towns (shoutout Fresno again) with low (\\$200) investment amount.

I started wondering how much money someone's making off all these residential air filters, so I started doing some data engineering on selections from the San Jaquin Valley. The results are interesting:

1. Incentives in the valley as a whole constitute a huge portion of all CA incentives. The numbers are rough, but around 11,500 of 14,000 incentives are in the valley, claiming around \\$300M of the \\$840M of investment funds.

2. San Jaquin Valley air filtration incentives in particular do represent a huge **number** of incentives (~5,500), but financially, the burden is quite low at around \\$1M. This makes sense when considering that basically all of these incentives are \\$200 air purifiers, while the first handful of incentives I'm looking at in the rural areas are ~\\$35,000 ag burning alternative initiatives, and there are just about as many of these as there are air filtration incentives.

### Side Project - Incentive Type Visualization

I split up CAPP incentives by unique types; unfortunately it looks like two different fields serve as identifiers here. When looking at the state as a whole, "Project Type" is not a good identifier because many, many projects don't have a project type; "Source Category" is a better way to identify the kind of work that's being done.

The exception here is the San Jaquin Valley; source categories here for the air filtration and agricultural burning alternative projects I IDed earlier are list simply as "Community Identified Project". I'm creating a new field "Project Kind" to run this logic. There are a few other, **very small** data subsets in this source category, many of them duplicates of project types, so without spending too much time cleaning up data, I'll only split out those subtypes if they are air filtration or ag burning alternatives. This ends up being much more manageable (14 symbol classes instead of 23, air filtration is even already listed as a source category).

A note for CARB - they should fix this or provide some documentation for the discrepancies.

Localized patterns are beginning to appear now. The San Jaquin patterns from before (including a few similar incentives near the Mexicali border), along with off-road agriculture are visible along the state's agricultural lines. Getting closer to the cities, off-road other types appear, followed by on-road, locomotive, and infrastructure projects once we zoom into the major cities. Finally, marine projects dominate the coasts.

The only other project kind I'm interested in but haven't addressed is stationary source projects. I was hoping this would be oil field electrification or something, but it looks sparse and all residential. I would say it might be residential electrification in rural areas where generators dominate, but \$2,000 for a household doesn't seem like it would do the trick for that.

Going point-to-point to estimate investment breakdown by project kind is a little tedious, I'll make a chart for it. It's actually very insightful, definitely a lesson in data visualization; for example, Infrastructure incentives represent only 240 of the ~14,000 total, but have received ~\\$175M of the ~\\$840M I mentioned earlier.

### Initial Analysis Shortcomings

At this point I'm realizing something valuable: Tract percentiles don't take into account the regional impact of investment. For example, incentive ID 4261 is a \\$2.5M infrastructure project in LA; it looks very unassuming because such a large percentile difference is standard in the way I'm viewing data right now; after all, there are no investments in the neighboring tract, so the percentile difference is basically -100, and the percentile difference in the tract containing that investment is ~30 (not that high but still).



### CAPP Incentive Buffer Zone Discussion

To fix the regional impact problem I identified, I want to create a project zone for each investment based on the investment amount in dollars. This needs to assume a few things:

1. Regional impact of an investment is a circle. This disregards any effects of air dispersion (ex. wind direction would favor impact downwind of pollution sources), as well as non-point project investments (ex. improvements on a stretch of highway would look more like a corridor).

2. ROI scales with cost. This disregards any effects of economies of scale.

3. Community impact is directly proportional to investment amount in dollars and constant at $125/person. This is the big one, and deserves some explanation. Some numbers can be found here:

    https://ww2.arb.ca.gov/news/carb-expands-eligibility-its-community-air-protection-program-funds-local-actions-improve-air
    
    CARB states that \\$500M of the program's funding has been used to advance equity, and that AB 617-selected communities represent 4M Californians. So. 500/4 gets us to \\$125/person. Obviously this is a huge leap in assumption, but the beauty of this project is that when the point is to identify underserved communities, the number doesn't matter all that much. We'll see later that the project zones don't actually extend that far based on my analysis, but really any number that doesn't cause the entire state to look severely overserved will still provide relative visualiztion of regional investment.

   This constant number assumes a constant ROI for all kinds of projects, which is also probably a stretch. Investigating which types of projects bring the highest ROI would be valuable, albeit a completely different analysis which wouldn't involve spatial data much if at all.

After writing everything out and problem solving some Python, I was faced with a data management problem: Iterative geoprocessing over tens of thousands of data points takes a long time (~10 hours by my calculations). To solve this, I started with a sample of the top 1% of incentives by investment amount, since these will be the most impactful. This still took some time, but the results were worth it: Project area based on my assumptions actually isn't that large. The bottom 99% of projects, using the \\$125/person assumption, generally aren't extending past one tract.

Some numbers on this: the bottom 99% start at ~\\$750,000. Tracts, according to the state of CA, are typically between 2,500 and 8,000 people. Using 5,000 people/tract as a reasonable average, it would take \\$625,000 of investment to affect an entire tract, very very close to the ~\\$750,000 investment cutoff I used for my geoprocessing.

### CAPP Incentives Buffer Zone Conclusions

I put together a few different layers and symbols to take a look at what's going on here. I did take a quick look at just the CalEnviroScreen pollution percentiles to ensure it matched up reasonably well with the CI score percentile; it did, good job CARB. With two different layers for CAPP incentives, one for the top 1% and one for the bottom 99%, I decided to use the "grant percentile - CI percentile" layer I put together earlier, along with those two CAPP layers, to visualize the data. 

The percentile difference adequately takes into account areas where lots of small investments occur in a single tract, plus those areas where, say, just one small investment occurs in a tract (like I said before, we're assuming impact of a small investment does not exceed the tract that it occurs in). In addition, the top 1% of CAPP projects as polygons lays over tracts that, looking just at the percentile difference, don't appear to have been affected by nearby investments. For example, tract 6037296210 in the LA Bay area is clearly affected by several of the marine investments in the Bay itself, where there are no people and no tracts.

The LA Bay area in particular is something I focused a lot on because of how interesting the "no people, lots of investment" looked. This is what cued me to the fact that, at least for these marine incentives, that $125/person assumption might be underestimating the reach of the projects. After all, all these projects barely even touch neighboring tracts despite being specifically sized to affect a proportional number of people.

Even with these limitations, a conclusion is clear. High CI scores are very much clustered in two places: South LA and the San Joaquin Valley. When looking at the percentile difference layer and CAPP point layers, however, you can see just how much investment has been put into the San Joaquin Valley and how adequately (or even over-) served the region is. In constrast, South LA still looks terrible. Despite some scattered projects (including the cluster of marine projects that should be further reviewed for impact), this area contains such a high relative population, with relatively such high CI scores and relatively so few incentives.

This trend can also be seen more faintly in the nearby areas of the San Fernando Valley and the I-10 corridor extending into San Bernadino/Riverside County, but the population in South LA specifically is still so dense compared to these areas.

### CAPP Incentives Buffer Zone Followup

So I wasn't happy with my analysis. While this started as an exercise in spatial analysis, in reality an air emission project's reach won't be a function of the project cost. The intensity of the impact probably will be, but, for example, a marginal reduction in emissions from a huge stack will have marginal benefits over the entire range of its air dispersion.

To gain some real insight, I'm turning to literature. Here's a decent paper: https://escholarship.org/content/qt26d1m24r/qt26d1m24r_noSplash_878074f45a8a6a721b68e2be7b927a10.pdf.

The region of study here is conveniently the same as mine, the San Pedro Bay, and the authors have done a lot of the AERMOD dispersion modeling I don't have time to do myself just to satisfy my curiosity. Their results are reasonably insightful: Ship emissions do cover quite a large area, and a lot of money has been invested in the San Pedro Bay area for shore power, reducing diesel engine emissions over a lot of South LA. However, roadway emissions still contribute more significantly to local emissions; roughly, maximum modeled concentrations in the study are 100 vs 400 ug/m^3 for ships and roadways, respectively, and that ratio only gets lower the farther you get from the port.

Targeting huge diesel engines makes sense from a money perspective. I was really hoping to find that what agencies are doing makes sense; after all, I'm just some guy and this is their job. This also offers perspective on the difficulty of reducing roadway emissions, primarily achieved through incentives like the Port of Long Beach's Clean Truck Program. This program, if it is even listed under CAPP, surely spans a huge radius, dispersing out with traffic from the port.

The question, then, is that even with these programs in place for quite a while, how do we address South LA's continued high CalEnviroScreen score? There's surely more research to be done on emission sources contributing to local air quality; are there more sources that need addressing? Are ships and diesel trucks coming from port still a huge contributor despite the work we've done on emission standards and electrification? Etc.

Finally, this is a good discussion point for global vs local emissions; while the political discourse has gotten complicated around mining and capitalism (shoutout to Elon Musk), this is a clear example of a huge economic incentive for vehicle electrification for the purposes of public health.